# Forward Propagation — How a Neural Network Predicts Output

*(CampusX 100 Days of Deep Learning, Day 9)*

---

## 1. Network architecture on the board

```
Input layer (4 features)      Layer 1 (3 neurons)     Layer 2 (2 neurons)     Output layer (1 neuron)

x_i1 ─┐                        ┌─ b11 ─┐
x_i2 ─┼──── fully connected ───┼─ b12 ─┼── fully connected ──┬─ b21 ─┐
x_i3 ─┤                        └─ b13 ─┘                     └─ b22 ─┴──── b31 ──→ ŷ
x_i4 ─┘
```

- Input: 4 features per row, e.g. $(x_{i1}, x_{i2}, x_{i3}, x_{i4})$
- **Layer 1**: 3 neurons — $b_{11}, b_{12}, b_{13}$
- **Layer 2**: 2 neurons — $b_{21}, b_{22}$
- **Output layer**: 1 neuron — $b_{31}$, whose activation *is* the final prediction: $O_{31} = \hat{y}$

Every neuron in a layer connects to **every** neuron in the next layer (fully connected / dense).

---

## 2. The core prediction formula (per neuron)

For any single neuron:

$$
\text{prediction} = \sigma(w^T x + b)
$$

i.e. take a weighted sum of inputs, add a bias, then squash it through the sigmoid so the raw linear combination becomes a bounded activation. This is applied **layer by layer** — the output of one layer becomes the input $x$ to the next.

---

## 3. Vectorizing Layer 1: from per-neuron to matrix form

Doing $w^Tx + b$ neuron-by-neuron (3 separate dot products for $b_{11}, b_{12}, b_{13}$) is what the board is replacing with **one matrix multiplication.**

**Weight matrix $W^1$** — one column per neuron in Layer 1, one row per input feature (**4×3**):

$$
W^1 =
\begin{bmatrix}
w'_{11} & w'_{12} & w'_{13} \\
w'_{21} & w'_{22} & w'_{23} \\
w'_{31} & w'_{32} & w'_{33} \\
w'_{41} & w'_{42} & w'_{43}
\end{bmatrix}_{4\times3}
$$

$w'_{jk}$ reads as: weight from input feature $j$ into neuron $k$ of layer 1.

**Input vector $X$** (**4×1**):

$$
X =
\begin{bmatrix} x_{i1} \\ x_{i2} \\ x_{i3} \\ x_{i4} \end{bmatrix}_{4\times1}
$$

**Bias vector $b^1$** (**3×1**, one bias per Layer-1 neuron):

$$
b^1 =
\begin{bmatrix} b_{11} \\ b_{12} \\ b_{13} \end{bmatrix}_{3\times1}
$$

**Putting it together — transpose $W^1$ so the dimensions line up for multiplication:**

$$
O^1 = (W^1)^T X + b^1
$$

Dimension check (exactly what's scribbled at the bottom of the board):

$$
\underbrace{(3\times4)}_{(W^1)^T} \cdot \underbrace{(4\times1)}_{X} \;+\; \underbrace{(3\times1)}_{b^1} \;=\; \underbrace{(3\times1)}_{O^1}
$$

$(W^1)^T$ turns the 4×3 weight matrix into 3×4, so multiplying by the 4×1 input gives a 3×1 result — one raw value per Layer-1 neuron — which then adds elementwise with the 3×1 bias vector to give $O^1$ (still pre-activation; apply $\sigma$ elementwise to get the actual layer-1 output that feeds Layer 2).

---

## 4. Generalizing to any layer $l$

$$
O^l = \sigma\big((W^l)^T \, O^{l-1} + b^l\big)
$$

where $O^0 = X$ (the raw input). Repeat this layer by layer:

$$
X \;\xrightarrow{\text{Layer 1}}\; O^1 \;\xrightarrow{\text{Layer 2}}\; O^2 \;\xrightarrow{\text{Layer 3}}\; O^3 = O_{31} = \hat{y}
$$

That final scalar $O_{31}$ **is** $\hat{y}$ — forward propagation is just this chain of (linear transform → bias add → sigmoid) applied layer after layer until you fall out the other end with a single prediction.

---

### One-line takeaway

> Forward propagation = repeatedly computing $\sigma((W^l)^T O^{l-1} + b^l)$ layer by layer; vectorizing it into matrices is purely a bookkeeping trick to do all of a layer's neurons in one matrix multiply instead of one dot product per neuron.

---

*Note: I skipped transcribing the numeric scratch-work (8.1/92/75/76/0 table, "6+2=8", "12+3" etc.) since those look like a specific worked example on the board rather than the general derivation — let me know if you want that example worked through too.*

# Forward Propagation
### How does a neural network predict output?
*CampusX — 100 Days of Deep Learning*

---

## 1. The big picture

Forward propagation is the process of pushing an input row **through every layer** of a network — linear combination → add bias → squash with an activation — until a single number pops out the other end: the prediction $\hat y$.

Nothing conceptually new happens per neuron; the whole video is really about **two things**:
1. What one neuron computes: $\sigma(w^Tx + b)$
2. How to stop doing that neuron-by-neuron and do a **whole layer in one matrix multiplication** instead.

---

## 2. Network used in the example

![Network architecture](network_architecture.svg)

| Layer | # neurons | Role |
|---|---|---|
| Input | 4 | raw features $x_{i1}, x_{i2}, x_{i3}, x_{i4}$ |
| Layer 1 | 3 | $b_{11}, b_{12}, b_{13}$ |
| Layer 2 | 2 | $b_{21}, b_{22}$ |
| Output | 1 | $b_{31}$ — its activation **is** $\hat y$ |

Every neuron connects to **every** neuron in the next layer (fully connected / dense network).

---

## 3. What a single neuron computes

$$
\text{prediction} = \sigma(w^Tx + b)
$$

- $w^Tx$ — dot product of the neuron's weight vector with its input vector
- $+ b$ — add the neuron's own bias (its personal offset)
- $\sigma(\cdot)$ — sigmoid squashes the raw sum into a bounded activation, which becomes the *input* to the next layer

Doing this once per neuron, per layer, is correct — but writing 3 separate dot products for Layer 1, then 2 more for Layer 2, gets tedious and doesn't scale. That's the motivation for vectorizing.

---

## 4. Vectorizing an entire layer

**Weight matrix for Layer 1** — one column per neuron, one row per input feature:

$$
W^1 =
\begin{bmatrix}
w'_{11} & w'_{12} & w'_{13} \\
w'_{21} & w'_{22} & w'_{23} \\
w'_{31} & w'_{32} & w'_{33} \\
w'_{41} & w'_{42} & w'_{43}
\end{bmatrix}_{4 \times 3}
$$

Read $w'_{jk}$ as *"weight from input feature $j$ into neuron $k$ of Layer 1."*

**Input vector** (4 × 1) and **bias vector** (3 × 1, one bias per Layer-1 neuron):

$$
X = \begin{bmatrix} x_{i1} \\ x_{i2} \\ x_{i3} \\ x_{i4} \end{bmatrix}_{4\times1}
\qquad
b^1 = \begin{bmatrix} b_{11} \\ b_{12} \\ b_{13} \end{bmatrix}_{3\times1}
$$

To make the shapes line up for multiplication, **transpose** the weight matrix:

$$
O^1 = (W^1)^T X + b^1
$$

![Matrix dimension flow](matrix_flow.svg)

$$
\underbrace{(3\times4)}_{(W^1)^T} \cdot \underbrace{(4\times1)}_{X} + \underbrace{(3\times1)}_{b^1} = \underbrace{(3\times1)}_{O^1}
$$

One matrix multiply now produces the raw (pre-activation) output of **all 3 neurons in Layer 1 simultaneously**. Apply $\sigma$ elementwise to $O^1$ and that becomes the input to Layer 2.

---

## 5. The general recurrence

$$
O^l = \sigma\big((W^l)^T\,O^{l-1} + b^l\big), \qquad O^0 = X
$$

Chained across the whole network:

$$
X \;\xrightarrow{\text{Layer 1}}\; O^1 \;\xrightarrow{\text{Layer 2}}\; O^2 \;\xrightarrow{\text{Output layer}}\; O^3 = \hat y
$$

That's forward propagation, end to end: **matrix multiply → add bias → activation, repeated once per layer**, until the last layer hands you the prediction.

---

## 6. Minimal NumPy implementation

```python
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward_propagation(X, weights, biases):
    """
    X        : input vector, shape (n_features, 1)
    weights  : list of W^l matrices, one per layer
    biases   : list of b^l vectors, one per layer
    """
    O = X
    for W, b in zip(weights, biases):
        Z = np.dot(W.T, O) + b   # (W^l)^T @ O^(l-1) + b^l
        O = sigmoid(Z)          # apply activation elementwise
    return O   # final O is y_hat

# Example dimensions matching the diagram above:
W1 = np.random.randn(4, 3)   # input(4) -> layer1(3)
W2 = np.random.randn(3, 2)   # layer1(3) -> layer2(2)
W3 = np.random.randn(2, 1)   # layer2(2) -> output(1)

b1 = np.zeros((3, 1))
b2 = np.zeros((2, 1))
b3 = np.zeros((1, 1))

X = np.array([[8.1], [92], [75], [76]])  # single input row, 4 features

y_hat = forward_propagation(X, [W1, W2, W3], [b1, b2, b3])
print("prediction:", y_hat)
```

Note the weight matrix shapes: `W1` is `(4,3)` — *(features from previous layer, neurons in this layer)* — matching $(W^1)$ before the transpose in the math above. `.T` inside the loop does the same job as the $(W^l)^T$ in the formula.

---

### One-line takeaway

> Forward propagation is just $\sigma(w^Tx+b)$ applied to a whole layer at once via matrix multiplication, then chained layer after layer until the last neuron's activation *is* $\hat y$.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 420" font-family="Consolas, monospace">
<rect width="760" height="420" fill="#0d1117"/>
<line x1="90" y1="70" x2="300" y2="100" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="70" x2="300" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="70" x2="300" y2="280" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="150" x2="300" y2="100" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="150" x2="300" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="150" x2="300" y2="280" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="230" x2="300" y2="100" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="230" x2="300" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="230" x2="300" y2="280" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="310" x2="300" y2="100" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="310" x2="300" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="90" y1="310" x2="300" y2="280" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="100" x2="500" y2="140" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="100" x2="500" y2="240" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="190" x2="500" y2="140" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="190" x2="500" y2="240" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="280" x2="500" y2="140" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="300" y1="280" x2="500" y2="240" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="500" y1="140" x2="670" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<line x1="500" y1="240" x2="670" y2="190" stroke="#8b949e" stroke-width="1.2" opacity="0.7"/>

<circle cx="90" cy="70" r="22" fill="#238636" stroke="#0d1117" stroke-width="2"/>
<text x="90" y="75" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">x_i1</text>

<circle cx="90" cy="150" r="22" fill="#238636" stroke="#0d1117" stroke-width="2"/>
<text x="90" y="155" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">x_i2</text>

<circle cx="90" cy="230" r="22" fill="#238636" stroke="#0d1117" stroke-width="2"/>
<text x="90" y="235" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">x_i3</text>

<circle cx="90" cy="310" r="22" fill="#238636" stroke="#0d1117" stroke-width="2"/>
<text x="90" y="315" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">x_i4</text>

<circle cx="300" cy="100" r="22" fill="#1f6feb" stroke="#0d1117" stroke-width="2"/>
<text x="300" y="105" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b11</text>

<circle cx="300" cy="190" r="22" fill="#1f6feb" stroke="#0d1117" stroke-width="2"/>
<text x="300" y="195" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b12</text>

<circle cx="300" cy="280" r="22" fill="#1f6feb" stroke="#0d1117" stroke-width="2"/>
<text x="300" y="285" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b13</text>

<circle cx="500" cy="140" r="22" fill="#8957e5" stroke="#0d1117" stroke-width="2"/>
<text x="500" y="145" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b21</text>

<circle cx="500" cy="240" r="22" fill="#8957e5" stroke="#0d1117" stroke-width="2"/>
<text x="500" y="245" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b22</text>

<circle cx="670" cy="190" r="22" fill="#da3633" stroke="#0d1117" stroke-width="2"/>
<text x="670" y="195" text-anchor="middle" font-size="14" font-family="Consolas, monospace" fill="white">b31</text>

<text x="90" y="40" text-anchor="middle" font-size="15" fill="#c9d1d9">Input (4)</text>
<text x="300" y="40" text-anchor="middle" font-size="15" fill="#c9d1d9">Layer 1 (3)</text>
<text x="500" y="40" text-anchor="middle" font-size="15" fill="#c9d1d9">Layer 2 (2)</text>
<text x="670" y="40" text-anchor="middle" font-size="15" fill="#c9d1d9">Output (1)</text>
<line x1="692" y1="190" x2="740" y2="190" stroke="#da3633" stroke-width="2" marker-end="url(#arrow)"/>
<text x="742" y="185" font-size="16" fill="#da3633">ŷ</text>
<defs>
<marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth">
<path d="M0,0 L0,6 L9,3 z" fill="#da3633" />
</marker>
</defs>
</svg>